# Adaptive TCT with Interleaved Non-TCT Queries

Demonstrates the benefit of **adaptivity** when TCT threshold queries are interleaved
with standard DP queries that charge ε to every active user.

**Setup:** every `INTERLEAVE_EVERY` TCT rounds, a flat non-TCT query charges `ε_extra`
to all active users (e.g., a histogram query, a model update).  This is exactly the
scenario covered by Theorem 5 of the paper: IDP filters remain valid when TCT queries
are interleaved with arbitrary DP mechanisms.

**Why adaptivity wins:** Under Adaptive TCT, users who rarely cross the threshold
spend almost nothing on TCT rounds — their budget is nearly fully available for
non-TCT queries.  Under Standard IDP, every TCT round also drains ε, compounding
the cost of non-TCT queries.

**Sections:**
1. Parameters, calibration, and upfront cost summary
2. Retention and budget curves for a single interleaving frequency
3. Robustness sweep — half-life vs interleaving frequency
4. Per-user cost stratification by hit count

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from simulation.population import User, UserPopulation
from simulation.query import compute_round_conversions, laplace_threshold_query
from simulation.accountants.adaptive_tct import AdaptiveTCTAccountant
from simulation.accountants.standard_dp import StandardDPAccountant
from simulation.calibration import q_from_epsilon, set_threshold_for_hit_rate
from simulation.datasets import generate_synthetic_population, PopulationConfig

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## Section 1: Parameters

In [ ]:
# ── Privacy / accountant parameters ──────────────────────────────────────────
EPSILON         = 0.01
DELTA           = 1e-6

# ── Non-TCT interleaving ─────────────────────────────────────────────────────
INTERLEAVE_EVERY = 10      # one non-TCT query every N TCT rounds
EPSILON_NONTCT   = EPSILON

# ── Population / simulation parameters ───────────────────────────────────────
N_USERS         = 300
N_ROUNDS        = 300
EPS_BUDGET      = 1.0
DELTA_BUDGET    = 1.0
TARGET_HIT_RATE = 0.05
POP_ALPHA       = 0.3
POP_BETA        = 3.0
SEED            = 42

q = q_from_epsilon(EPSILON)

config = PopulationConfig(
    n_users=N_USERS, alpha=POP_ALPHA, beta=POP_BETA,
    epsilon_budget=EPS_BUDGET, delta_budget=DELTA_BUDGET,
)

calib_pop = generate_synthetic_population(config, rng=np.random.default_rng(SEED))
THRESHOLD, actual_hr = set_threshold_for_hit_rate(
    calib_pop, epsilon=EPSILON, target_hit_rate=TARGET_HIT_RATE,
    n_pilot=2000, rng=np.random.default_rng(SEED + 9999),
)

acc_adap = AdaptiveTCTAccountant(EPSILON, DELTA)

print(f'ε={EPSILON}, δ={DELTA}, q={q:.4f}')
print(f'Threshold T={THRESHOLD:.3f},  empirical hit rate={actual_hr:.3f}')
print(f'Interleaving: 1 non-TCT query (ε={EPSILON_NONTCT}) every {INTERLEAVE_EVERY} TCT rounds')
print()
print('Upfront costs:')
print(f'  Adaptive TCT  K_1·ε = {acc_adap.compositions(1) * EPSILON:.3f}  (K_1={acc_adap.compositions(1):.1f})')
print(f'  IDP           ε     = {EPSILON:.3f}  (per TCT round)')
print()
non_tct_rounds = N_ROUNDS // INTERLEAVE_EVERY
print(f'Over {N_ROUNDS} rounds: {non_tct_rounds} non-TCT queries → {non_tct_rounds * EPSILON_NONTCT:.2f}ε flat cost per user')

## Section 2: Single Run — Retention and Budget Curves

In [ ]:
def run_interleaved(accountant, interleave_every, eps_nontct, seed=SEED):
    """
    Run the simulation with periodic flat non-TCT charges.

    Every `interleave_every` TCT rounds, charge `eps_nontct` to all active users
    (non-TCT query — no threshold, no noise, every user pays regardless of hits).

    Returns:
        active_frac   list[float]  active fraction after each round
        avg_eps       list[float]  mean ε remaining among active users after each round
        hit_counts    dict[user_id -> int]  actual TCT hit count per user at end
        final_eps     dict[user_id -> float]  final ε remaining per user
    """
    rng = np.random.default_rng(seed)
    pop = generate_synthetic_population(config, rng=np.random.default_rng(seed))

    active_frac = []
    avg_eps     = []

    for r in range(N_ROUNDS):
        active = pop.active_users
        if not active:
            break

        # ── TCT round ────────────────────────────────────────────────────────
        true_count, contributions = compute_round_conversions(active, rng)
        is_top, _ = laplace_threshold_query(true_count, THRESHOLD, EPSILON, rng)

        for i, user in enumerate(active):
            e, d = accountant.charge(
                user=user, is_top=is_top,
                contributed=contributions[i], round_num=r,
            )
            user.epsilon_spent += e
            user.delta_spent   += d

        # ── Non-TCT query every INTERLEAVE_EVERY rounds ───────────────────────
        if interleave_every > 0 and (r + 1) % interleave_every == 0:
            for user in pop.active_users:
                user.epsilon_spent += eps_nontct

        pop.drop_depleted()

        eps_dist = pop.epsilon_remaining_distribution()
        active_frac.append(pop.active_fraction)
        avg_eps.append(float(np.mean(eps_dist)) if eps_dist else 0.0)

    hit_counts = {u.user_id: accountant.hit_count(u.user_id) for u in pop.all_users}
    final_eps  = {u.user_id: max(u.epsilon_remaining, 0.0) for u in pop.all_users}

    return active_frac, avg_eps, hit_counts, final_eps


def half_life(active_frac):
    """Round where active fraction first drops to or below 0.5."""
    for r, f in enumerate(active_frac):
        if f <= 0.5:
            return r
    return len(active_frac)


print('Running simulations...')
adap_active, adap_eps, adap_hits, adap_final = run_interleaved(
    AdaptiveTCTAccountant(EPSILON, DELTA), INTERLEAVE_EVERY, EPSILON_NONTCT)
idp_active,  idp_eps,  idp_hits,  idp_final  = run_interleaved(
    StandardDPAccountant(epsilon_per_round=EPSILON, delta_per_round=0.0),
    INTERLEAVE_EVERY, EPSILON_NONTCT)
print('Done.')

In [ ]:
def pad(curve, n):
    return list(curve) + [curve[-1]] * (n - len(curve))

n = max(len(adap_active), len(idp_active)) + 1
rnds = list(range(n))

adap_a = pad(adap_active, n)
idp_a  = pad(idp_active,  n)
adap_e = pad(adap_eps, n)
idp_e  = pad(idp_eps,  n)

nontct_rounds = [r for r in range(N_ROUNDS) if (r + 1) % INTERLEAVE_EVERY == 0]

hl_adap = half_life(adap_active)
hl_idp  = half_life(idp_active)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(rnds, adap_a, color='steelblue', lw=2, label=f'Adaptive TCT  (half-life={hl_adap})')
ax.plot(rnds, idp_a,  color='gray',      lw=2, linestyle='--', label=f'Standard IDP  (half-life={hl_idp})')
ax.axhline(0.5, color='silver', lw=1, linestyle=':')
for r in nontct_rounds[:5]:
    ax.axvline(r, color='mediumpurple', lw=0.6, alpha=0.4)
ax.set_xlabel('Round')
ax.set_ylabel('Fraction of users active')
ax.set_title('User Retention')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(rnds, adap_e, color='steelblue', lw=2, label='Adaptive TCT')
ax.plot(rnds, idp_e,  color='gray',      lw=2, linestyle='--', label='Standard IDP')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('Round')
ax.set_ylabel('Avg ε remaining (active users)')
ax.set_title('Average Budget Remaining')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(
    f'N={N_USERS}, ε={EPSILON}, δ={DELTA}  |  '
    f'non-TCT query every {INTERLEAVE_EVERY} rounds (ε={EPSILON_NONTCT})  |  '
    f'purple ticks = non-TCT rounds',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'Half-life gap (Adaptive TCT − IDP): {hl_adap - hl_idp:+d} rounds')

## Section 3: Robustness Sweep — Half-Life vs Interleaving Frequency

How does the adaptivity advantage change as non-TCT queries become more or less frequent?

- **High interleaving** (small `interleave_every`): non-TCT queries dominate — all accountants
  deplete at nearly the same rate (flat cost overwhelms TCT savings)
- **Low interleaving** (large `interleave_every`): TCT rounds dominate — full H1 advantage visible
- **No interleaving** (`interleave_every=∞`): pure TCT comparison, reproduces H1 result

In [ ]:
interleave_sweep = [2, 5, 10, 20, 50, 0]  # 0 = no non-TCT queries
labels = [str(v) if v > 0 else '∞' for v in interleave_sweep]

hl_adap_sweep = []
hl_idp_sweep  = []

for iv in interleave_sweep:
    a_act, *_ = run_interleaved(AdaptiveTCTAccountant(EPSILON, DELTA), iv, EPSILON_NONTCT)
    i_act, *_ = run_interleaved(
        StandardDPAccountant(epsilon_per_round=EPSILON, delta_per_round=0.0), iv, EPSILON_NONTCT)
    hl_adap_sweep.append(half_life(a_act))
    hl_idp_sweep.append(half_life(i_act))

x = range(len(interleave_sweep))
gap = [hl_adap_sweep[i] - hl_idp_sweep[i] for i in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x, hl_adap_sweep, 'o-', color='steelblue', lw=2, label='Adaptive TCT')
ax.plot(x, hl_idp_sweep,  '^--', color='gray',     lw=2, label='Standard IDP')
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_xlabel('Non-TCT query every N rounds  (∞ = no interleaving)')
ax.set_ylabel('Half-life (rounds)')
ax.set_title('Half-Life vs Interleaving Frequency')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x, gap, 'o-', color='steelblue', lw=2)
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_xlabel('Non-TCT query every N rounds  (∞ = no interleaving)')
ax.set_ylabel('Half-life advantage (rounds)')
ax.set_title('Adaptive TCT − IDP Half-Life Advantage')
ax.grid(True, alpha=0.3)

plt.suptitle(f'ε={EPSILON}, δ={DELTA}, ε_nontct={EPSILON_NONTCT}', fontsize=11)
plt.tight_layout()
plt.show()

header = '{:>12}  {:>8}  {:>8}  {:>10}'.format('Interleave', 'Adap HL', 'IDP HL', 'Adap-IDP')
print(header)
print('-' * 44)
for i, lbl in enumerate(labels):
    print(f'{lbl:>12}  {hl_adap_sweep[i]:>8}  {hl_idp_sweep[i]:>8}  {gap[i]:>+10}')

## Section 4: Per-User Cost Stratification by Hit Count

The adaptivity claim is not just that the population average improves — it is that
**privacy cost concentrates on users who actually contribute to target hits**.  Users
with zero or one hit should retain most of their budget; heavy contributors pay more.

Under Standard TCT all users pay K_τ·ε at registration regardless of actual hits.
Under Adaptive TCT users with fewer hits pay less.

In [ ]:
adap_active2, adap_eps2, adap_hits2, adap_final2 = run_interleaved(
    AdaptiveTCTAccountant(EPSILON, DELTA), INTERLEAVE_EVERY, EPSILON_NONTCT)

def stratify(hits_dict, final_eps_dict):
    buckets = defaultdict(list)
    for uid, hits in hits_dict.items():
        bucket = min(hits, 3)
        buckets[bucket].append(final_eps_dict[uid])
    return buckets

adap_buckets = stratify(adap_hits2, adap_final2)
idp_buckets  = stratify(idp_hits,   idp_final)

bucket_labels = ['0 hits', '1 hit', '2 hits', '3+ hits']
bucket_keys   = [0, 1, 2, 3]

adap_means = [np.mean(adap_buckets[k]) if adap_buckets[k] else 0.0 for k in bucket_keys]
idp_means  = [np.mean(idp_buckets[k])  if idp_buckets[k]  else 0.0 for k in bucket_keys]
adap_ns    = [len(adap_buckets[k]) for k in bucket_keys]

x_pos = np.arange(len(bucket_keys))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars_adap = ax.bar(x_pos - width/2, adap_means, width, color='steelblue', alpha=0.85, label='Adaptive TCT')
bars_idp  = ax.bar(x_pos + width/2, idp_means,  width, color='gray',      alpha=0.85, label='Standard IDP')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'{lbl}\n(n={adap_ns[i]})' for i, lbl in enumerate(bucket_labels)])
ax.set_ylabel('Mean ε remaining at end of simulation')
ax.set_title('Budget Remaining by Hit Count Bucket')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(0, color='black', lw=0.5)
for bar in bars_adap:
    h = bar.get_height()
    if h > 0.1:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
                ha='center', va='bottom', fontsize=8, color='steelblue')
for bar in bars_idp:
    h = bar.get_height()
    if h > 0.1:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}',
                ha='center', va='bottom', fontsize=8, color='gray')

ax = axes[1]
advantage = [adap_means[i] - idp_means[i] for i in range(len(bucket_keys))]
colors = ['steelblue' if v >= 0 else 'salmon' for v in advantage]
ax.bar(x_pos, advantage, color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(bucket_labels)
ax.set_ylabel('Δ ε remaining (Adaptive TCT − IDP)')
ax.set_title('Adaptive TCT Budget Advantage by Hit Count')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    f'N={N_USERS}, ε={EPSILON}, δ={DELTA}  |  '
    f'non-TCT query every {INTERLEAVE_EVERY} rounds  |  budget={EPS_BUDGET}',
    fontsize=10
)
plt.tight_layout()
plt.show()

print(f'\n{"Bucket":<10}  {"n":>6}  {"Adap mean ε":>12}  {"IDP mean ε":>11}  {"Advantage":>10}')
print('-' * 54)
for i, lbl in enumerate(bucket_labels):
    print(f'{lbl:<10}  {adap_ns[i]:>6}  {adap_means[i]:>12.3f}  '
          f'{idp_means[i]:>11.3f}  {advantage[i]:>+10.3f}')